In [4]:
# If there's a conflict run this command first

# !pip uninstall -y chromadb

In [1]:
%pip install chromadb langchain-core langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
import posthog, deepeval, chromadb
print("posthog", posthog.__version__)
print("deepeval", deepeval.__version__)
print("chromadb", chromadb.__version__)

posthog 5.4.0
deepeval 3.6.9
chromadb 1.3.4


In [3]:
import json, random, math, csv, os, time
from collections import defaultdict
from statistics import quantiles
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import FiltrationConfig, StylingConfig, ContextConstructionConfig

random.seed(42)

SNAPSHOT = "test_apartments.json"
OUT_CSV = "golden_test_rag_style.csv"

def fmt_listing_line(r):
    addr = r.get("address","").strip()
    nbh  = r.get("neighborhood","").strip()
    price= r.get("price")
    beds = r.get("bedrooms"); baths = r.get("bathrooms")
    ams  = r.get("amenities") or []
    parts = []
    if addr or nbh: parts.append(f"{addr} — {nbh}".strip(" —"))
    meta = []
    if beds is not None:  meta.append(f"{beds} BR")
    if baths is not None: meta.append(f"{baths} BA")
    if price is not None: meta.append(f"${price:,}")
    if meta: parts.append(" / ".join(meta))
    if ams:  parts.append("Amenities: " + ", ".join(map(str, ams[:6])))
    return " | ".join(parts)

def build_context_block(listings):
    # mimic your “Top K listings retrieved for this turn” text block
    lines = []
    for i, r in enumerate(listings, 1):
        line = fmt_listing_line(r)
        if not line: 
            continue
        lines.append(f"{i}. {line}")
    return "\n".join(lines)

def to_price_band(p, qs):
    if p is None: return "unknown"
    q1, q2, q3 = qs
    if p <= q1: return "low"
    if p <= q2: return "mid"
    if p <= q3: return "upper"
    return "high"


In [4]:
# load data and stratify
with open(SNAPSHOT) as f:
    data = json.load(f)

prices = sorted([r["price"] for r in data if isinstance(r.get("price"), (int, float))])
qs = quantiles(prices, n=4) if len(prices) >= 4 else [min(prices or [0])]*3

buckets = defaultdict(list)
for r in data:
    ctx = fmt_listing_line(r)
    if not ctx: 
        continue
    nbh = (r.get("neighborhood") or "unknown").lower()
    pb  = to_price_band(r.get("price"), qs)
    buckets[(nbh, pb)].append(r)

In [5]:
# sample top-k blocks per stratum
TOP_K = 5
TARGET_GOLDENS = 300
MAX_PER_BUCKET = max(2, TARGET_GOLDENS // max(1,len(buckets)))

candidate_blocks = []
for key, recs in buckets.items():
    random.shuffle(recs)
    # take multiple disjoint blocks of size TOP_K
    for i in range(0, min(len(recs), MAX_PER_BUCKET*TOP_K), TOP_K):
        block = recs[i:i+TOP_K]
        if len(block) == TOP_K:
            candidate_blocks.append({
                "listings": block,
                "tags": [f"nbh:{key[0]}", f"price:{key[1]}"]
            })

random.shuffle(candidate_blocks)
candidate_blocks = candidate_blocks[:TARGET_GOLDENS]
len(candidate_blocks)

290

In [ ]:
from deepeval.synthesizer import Synthesizer
from tenacity import RetryError

try:
    from openai import RateLimitError
except ImportError:  # fallback when openai pkg not installed in this kernel
    class RateLimitError(Exception):
        pass

BATCH_SIZE = 10            # contexts per outer batch
PER_CONTEXT_DELAY = 3.0    # seconds between individual context generations
COOLDOWN_SECONDS = 90      # pause between batches
MAX_RETRIES_PER_CONTEXT = 5

def listing_str(rec):
    parts = []
    addr = rec.get("address", "").strip()
    nbh = (rec.get("neighborhood") or "").strip()
    if addr or nbh:
        parts.append(f"{addr} — {nbh}".strip())
    meta = []
    bedrooms = rec.get("bedrooms")
    bathrooms = rec.get("bathrooms")
    price = rec.get("price")
    if bedrooms is not None:
        meta.append(f"{bedrooms} BR")
    if bathrooms is not None:
        meta.append(f"{bathrooms} BA")
    if price is not None:
        meta.append(f"${price:,}")
    if meta:
        parts.append(" / ".join(meta))
    return " | ".join(parts)

contexts = []
source_files = []

for block in candidate_blocks:
    listings = block["listings"]
    rendered = []
    ids = []
    for rec in listings:
        ids.append(str(rec.get("id", "")))
        line = listing_str(rec)
        if line:
            rendered.append(line)
    if len(rendered) >= 2:
        contexts.append(rendered)
        source_files.append(",".join(ids))

assert len(contexts) == len(source_files)
context_pairs = list(zip(contexts, source_files))
total_contexts = len(context_pairs)


def wait_with_backoff(base_sleep, attempt):
    wait = base_sleep * max(1, attempt)
    print(f"   ↳ Rate limit hit; backing off for {wait:.1f}s (attempt {attempt})...")
    time.sleep(wait)


def run_context_with_retries(synth, ctx, src, ctx_idx):
    attempt = 0
    while True:
        try:
            synth.generate_goldens_from_contexts(
                contexts=[ctx],
                include_expected_output=True,
                max_goldens_per_context=1,
                source_files=[src],
            )
            break
        except RateLimitError as err:
            attempt += 1
            if attempt > MAX_RETRIES_PER_CONTEXT:
                raise
            print(f"[WARN] OpenAI rate limit while building context {ctx_idx}/{total_contexts}: {err}")
            wait_with_backoff(COOLDOWN_SECONDS, attempt)
        except RetryError as err:
            attempt += 1
            if attempt > MAX_RETRIES_PER_CONTEXT:
                raise
            underlying = err.last_attempt.exception() if err.last_attempt else err
            print(f"[WARN] Deepeval retry exhausted on context {ctx_idx}/{total_contexts}: {underlying}")
            wait_with_backoff(COOLDOWN_SECONDS, attempt)


def iter_batches(pairs, size):
    for start in range(0, len(pairs), size):
        yield start, pairs[start:start + size]


synth = Synthesizer()

for batch_start, batch in iter_batches(context_pairs, BATCH_SIZE):
    batch_end = batch_start + len(batch)
    print(f"Processing contexts {batch_start + 1}-{batch_end} of {total_contexts}...")
    for offset, (ctx, src) in enumerate(batch, start=1):
        global_idx = batch_start + offset
        print(f"  ↳ Generating golden for context {global_idx}/{total_contexts}")
        run_context_with_retries(synth, ctx, src, global_idx)
        if PER_CONTEXT_DELAY and global_idx < total_contexts:
            time.sleep(PER_CONTEXT_DELAY)
    if batch_end < total_contexts:
        print(f"Batch complete, sleeping {COOLDOWN_SECONDS}s before next batch...")
        time.sleep(COOLDOWN_SECONDS)


Output()

Generating goldens batch 1/6 (50 contexts)...


[WARN] Deepeval retry exhausted on batch 1/6: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-fZ1vvf6i1BF1c9APb07AU6zO on tokens per min (TPM): Limit 30000, Used 30000, Requested 825. Please try again in 1.65s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
   ↳ Rate limit hit; backing off for 65s (attempt 1)...


Output()

[WARN] Deepeval retry exhausted on batch 1/6: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-fZ1vvf6i1BF1c9APb07AU6zO on tokens per min (TPM): Limit 30000, Used 30000, Requested 424. Please try again in 848ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
   ↳ Rate limit hit; backing off for 130s (attempt 2)...


KeyboardInterrupt: 

Output()

Generating goldens batch 1/6 (50 contexts)...


[WARN] Deepeval retry exhausted on batch 1/6: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1 in organization org-fZ1vvf6i1BF1c9APb07AU6zO on tokens per min (TPM): Limit 30000, Used 30000, Requested 417. Please try again in 834ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
   ↳ Rate limit hit; backing off for 65s (attempt 1)...


In [ ]:
synth = Synthesizer()
synth.generate_goldens_from_docs(
    document_paths=["docs/a.txt","docs/b.md","docs/c.pdf"],   # required
    include_expected_output=True,                             # optional (default True)
    max_goldens_per_context=2,                                # optional (default 2)
    context_construction_config=ContextConstructionConfig(    # optional
        max_contexts_per_document=2,
        max_context_length=3,
        chunk_size=800,
        chunk_overlap=0
    )
)